In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# -------------------------
# Load datasets
# -------------------------
train = pd.read_parquet('../data/model/final_train.parquet')
val   = pd.read_parquet('../data/model/final_val.parquet')
test  = pd.read_parquet('../data/model/final_test.parquet')

train = train.sort_values(["tic", "Date"])
val   = val.sort_values(["tic", "Date"])
test  = test.sort_values(["tic", "Date"])

# -------------------------
# Feature groups
# -------------------------
fund_features = [
    "sales_growth_qoq", "sales_growth_ttm", "asset_growth", "equity_growth",
    "roa_ttm", "roe_ttm", "gross_margin_ttm", "oper_margin_ttm",
    "net_margin_ttm", "log_mktcap", "bm", "earnings_yield", "cf_yield",
    "sales_yield", "div_yield", "leverage", "current_ratio", "cash_assets",
    "accruals_ta"
]

market_features = ["Open", "High", "Low", "Close", "Volume"]

macro_features = [
    "cpi", "fedfunds", "industrial_production", "gdp", "retail_sales",
    "unemployment", "t10y", "t2y", "t3m", "aaa_yield", "vix", "sp500",
    "yield_spread_10y_2y"
]

emb_cols = [c for c in train.columns if c.startswith("pca_emb_")]
news_features = ["mean_sentiment", "max_sentiment", "min_sentiment",
                 "sum_sentiment", "news_count"]

feature_groups = {
    "market": market_features,
    "fundamental": fund_features,
    "macro": macro_features,
    "news": news_features,
    "market+fund": market_features + fund_features,
    "market+macro": market_features + macro_features,
    "market+news": market_features + news_features,
    "ALL": market_features + fund_features + macro_features + news_features
}

# -------------------------
# Utility metrics
# -------------------------
def rmse(y_true, y_pred):
    mask = ~np.isnan(y_true)
    return np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))

def direction_accuracy(y_true, y_pred):
    mask = (~np.isnan(y_true))
    return (np.sign(y_true[mask]) == np.sign(y_pred[mask])).mean()

y_train = train["return_next_day"].values
y_val   = val["return_next_day"].values
y_test  = test["return_next_day"].values

# -------------------------
# Loop through feature groups
# -------------------------
results = []

for name, feats in feature_groups.items():
    print(f"Training model: {name} ({len(feats)} features)")

    X_train = train[feats]
    X_val   = val[feats]
    X_test  = test[feats]

    model = LinearRegression()
    model.fit(X_train, y_train)

    val_pred  = model.predict(X_val)
    test_pred = model.predict(X_test)

    results.append({
        "feature_group": name,
        "num_features": len(feats),
        "val_rmse": rmse(y_val, val_pred),
        "test_rmse": rmse(y_test, test_pred),
        "val_da": direction_accuracy(y_val, val_pred),
        "test_da": direction_accuracy(y_test, test_pred)
    })

# -------------------------
# Summary table
# -------------------------
results_df = pd.DataFrame(results)
display(results_df.sort_values("val_rmse"))


In [ ]:
plt.figure(figsize=(12,5))
plt.bar(results_df["feature_group"], results_df["val_rmse"])
plt.xticks(rotation=45)
plt.ylabel("Val RMSE")
plt.title("RMSE by Feature Group (Linear Model)")
plt.show()


In [ ]:
plt.figure(figsize=(12,5))
plt.bar(results_df["feature_group"], results_df["val_da"])
plt.xticks(rotation=45)
plt.ylabel("Val Directional Accuracy")
plt.title("Directional Accuracy by Feature Group (Linear Model)")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# ============ Utility Metrics ============
def rmse(y_true, y_pred):
    mask = ~np.isnan(y_true)
    return np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))

def direction_accuracy(y_true, y_pred):
    mask = ~np.isnan(y_true)
    return (np.sign(y_true[mask]) == np.sign(y_pred[mask])).mean()

# ============ Load Data ============
train = pd.read_parquet('../data/model/final_train.parquet')
val   = pd.read_parquet('../data/model/final_val.parquet')
test  = pd.read_parquet('../data/model/final_test.parquet')

train = train.sort_values(["tic", "Date"])
val   = val.sort_values(["tic", "Date"])
test  = test.sort_values(["tic", "Date"])

y_train = train["return_next_day"].values
y_val   = val["return_next_day"].values
y_test  = test["return_next_day"].values

# ============ Model Comparison ============
results = []

for name, feats in feature_groups.items():
    X_train = train[feats]
    X_val   = val[feats]
    X_test  = test[feats]

    print(f"\n==== Feature Group: {name} ({len(feats)} features) ====")

    # -------- Random Forest --------
    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    )
    rf.fit(X_train, y_train)
    val_pred_rf  = rf.predict(X_val)
    test_pred_rf = rf.predict(X_test)

    # -------- XGBoost --------
    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        n_jobs=-1,
        tree_method="hist"
    )
    xgb.fit(X_train, y_train)
    val_pred_xgb  = xgb.predict(X_val)
    test_pred_xgb = xgb.predict(X_test)

    results.append({
        "feature_group": name,
        "model": "RF",
        "val_rmse": rmse(y_val, val_pred_rf),
        "test_rmse": rmse(y_test, test_pred_rf),
        "val_da": direction_accuracy(y_val, val_pred_rf),
        "test_da": direction_accuracy(y_test, test_pred_rf)
    })

    results.append({
        "feature_group": name,
        "model": "XGB",
        "val_rmse": rmse(y_val, val_pred_xgb),
        "test_rmse": rmse(y_test, test_pred_xgb),
        "val_da": direction_accuracy(y_val, val_pred_xgb),
        "test_da": direction_accuracy(y_test, test_pred_xgb)
    })
    print(f"RF Val RMSE: {results[-2]['val_rmse']:.6f}, XGB Val RMSE: {results[-1]['val_rmse']:.6f}")
    print(f"RF Test RMSE: {results[-2]['test_rmse']:.6f}, XGB Test RMSE: {results[-1]['test_rmse']:.6f}")
    print(f"RF Val DA: {results[-2]['val_da']:.6f}, XGB Val DA: {results[-1]['val_da']:.6f}")
    print(f"RF Test DA: {results[-2]['test_da']:.6f}, XGB Test DA: {results[-1]['test_da']:.6f}")

# Summary table
results_df = pd.DataFrame(results)
display(results_df.sort_values(["feature_group", "model"]))


In [15]:
from sklearn.linear_model import LinearRegression

ols = LinearRegression()
ols.fit(X_train, y_train)

val_pred_ols  = ols.predict(X_val)
test_pred_ols = ols.predict(X_test)

import numpy as np

# RMSE
val_rmse  = rmse(y_val.values,  val_pred_ols)
test_rmse = rmse(y_test.values, test_pred_ols)

val_da  = direction_accuracy(y_val.values, val_pred_ols)
test_da = direction_accuracy(y_test.values, test_pred_ols)

print(val_rmse, test_rmse)
print(val_da, test_da)




0.03914045041732376 0.039235862655955425
0.493150135875011 0.5219396749498901
